# CNN 1D com fusão de superpixels: a otimização do trabalho de Souza (2023)

Este notebook só chama o código do pacote `src/hsi/`, uma etapa por seção:

| | Etapa | O que a célula mostra | Código |
| --- | --- | --- | --- |
| 1 | Configuração e ambiente | versões, dispositivo e modo determinístico | `config.py`, `ambiente.py` |
| 2 | Dados e segmentação | a cena em RGB e as regiões de SEEDS, LSC e Watershed | `dados.py`, `segmentadores.py` |
| 3 | CNN 1D com fusão | as camadas e o tamanho da rede | `modelo.py` |
| 4 | Treino | a linha de base em Indian Pines e as curvas | `treino.py`, `pipeline.py` |
| 5 | Busca de hiperparâmetros | uma busca bayesiana curta | `hpo.py` |
| 6 | Fusão concorrente | o esquema do orientador contra a inferência em lote | `paralelismo/fusao_concorrente.py` |
| 7 | Partição da cena | treino em N partes, em threads ou processos | `paralelismo/particao.py` |
| 8 | Experimentos e figuras | as campanhas da monografia | `experimentos/`, `figuras.py` |

As variáveis de execução (padrões de treino, parâmetros dos segmentadores, espaço da busca e o desenho de cada experimento) ficam em `config.toml`, na raiz do repositório.

Para preparar o ambiente:

```bash
uv venv --python 3.12
uv pip install -r requirements.txt
```

As cenas (`.mat`) ficam em `data/dataHSI/`. As células que demoram minutos começam desligadas.

In [ ]:
import sys
from pathlib import Path

# O pacote mora em src/; edições nele passam a valer sem reiniciar o kernel.
sys.path.insert(0, str(Path.cwd() / "src"))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import torch

## 1. Configuração e ambiente

`ExperimentConfig` guarda tudo o que define uma execução; os padrões vêm da seção `[execucao]` de `config.toml` e são os hiperparâmetros do original (lote 32, taxa 0,001, dropout 0,5, SEEDS) com a CNN do projeto. `configure_determinism` fixa o que poderia mudar o resultado entre execuções com a mesma semente.

In [ ]:
from hsi.ambiente import check_versions, configure_determinism, select_device
from hsi.config import SEGMENTERS, ExperimentConfig

check_versions()
device = select_device()
configure_determinism()

## 2. Dados e segmentação

Dois ramos. O espectral reduz cada pixel por PCA (30 componentes em Indian Pines, 15 nas outras cenas). O espacial gera uma composição RGB com três componentes e a segmenta; o id da região de cada pixel é o que a CNN recebe. LSC e Watershed estão calibrados para gerar o mesmo número de regiões que o SEEDS. A divisão é a do original: 85% treino, 7,5% validação e 7,5% teste.

In [ ]:
from hsi.config import SEMENTE_DADOS
from hsi.dados import apply_pca, load_scene, pixel_table
from hsi.segmentadores import SEGMENTER_PARAMS, rgb_composite, segment

cubo, gt = load_scene("IP")
pixels, _ = pixel_table(cubo, gt)
rgb = rgb_composite(apply_pca(pixels, 3, SEMENTE_DADOS), gt.shape)

fig, eixos = plt.subplots(1, 1 + len(SEGMENTERS), figsize=(12, 3.4))
eixos[0].imshow(rgb)
eixos[0].set_title("composição RGB")
for eixo, segmentador in zip(eixos[1:], SEGMENTERS):
    regioes = segment(rgb, segmentador, SEGMENTER_PARAMS[segmentador]["IP"])
    borda = np.zeros(regioes.shape, dtype=bool)
    borda[1:] |= regioes[1:] != regioes[:-1]
    borda[:, 1:] |= regioes[:, 1:] != regioes[:, :-1]
    eixo.imshow(np.where(borda[..., None], 255, rgb).astype(np.uint8))
    eixo.set_title(f"{segmentador}: {len(np.unique(regioes))} regiões")
for eixo in eixos:
    eixo.axis("off")
plt.show()

## 3. CNN 1D com fusão

Quatro convoluções sobre o espectro do pixel (16, 32, 64 e 128 filtros, *max-pooling* depois das três primeiras). O id do superpixel é concatenado ao vetor achatado, e só as camadas densas (256 e 128) veem as duas informações. A CNN do notebook de Souza (2023) continua disponível com `cnn="original"`.

In [ ]:
from hsi.modelo import CNN1D

exemplo = CNN1D(n_components=30, n_classes=16, dropout=0.5)
print(exemplo)
saida = exemplo.convolucoes(torch.zeros(1, 1, 30)).shape[1]
print(f"{sum(p.numel() for p in exemplo.parameters()):,} parâmetros; as densas recebem "
      f"{saida} características + 1 id")

## 4. Treino

Adam, entropia cruzada e parada depois de 10 épocas sem a acurácia de validação melhorar 0,001, com os melhores pesos restaurados. `run_experiment` vai da cena às métricas de teste (OA, AA e Kappa). A célula abaixo é a linha de base em Indian Pines (cerca de 1 min).

In [ ]:
from hsi.pipeline import run_experiment

config = ExperimentConfig(dataset="IP")
resultado = run_experiment(config, verbose=True)

historico = resultado["historico"]
fig, (perda, acuracia) = plt.subplots(1, 2, figsize=(10, 3.4))
for eixo, chave, nome in ((perda, "loss", "perda"), (acuracia, "accuracy", "acurácia")):
    eixo.plot(historico[chave], label="treino")
    eixo.plot(historico[f"val_{chave}"], label="validação")
    eixo.set(xlabel="época", ylabel=nome)
acuracia.legend()
plt.show()

## 5. Busca de hiperparâmetros

As duas estratégias do projeto. A busca em grade (`strategy="grid"`) cobre segmentador × taxa de aprendizado × dropout (27 pontos); a otimização bayesiana (`strategy="bayes"`), com processo gaussiano, percorre o espaço inteiro (também segmentação, weight decay, lote e PCA). Cada trial vale pela melhor acurácia de validação; o teste só é relatado.

In [ ]:
BUSCAR = False          # 8 trials curtos, cerca de 1 min

if BUSCAR:
    from hsi.hpo import automated_hpo

    base = ExperimentConfig(dataset="IP", train_fraction=0.10)
    trials = automated_hpo(base, n_trials=8, search_epochs=10, strategy="bayes")
    colunas = ["trial", "segmentador", "learning rate", "dropout", "batch", "PCA",
               "val_accuracy (objetivo)"]
    display(trials[colunas])

## 6. Fusão concorrente

O esquema do orientador. A thread T0 segmenta a cena enquanto T1..TN passam os pixels pelas convoluções, que não dependem do mapa; cada tarefa espera o mapa só para ler o id e concatenar. Vale para a inferência de um modelo treinado. A célula usa o modelo da seção 4 e confere que as classes saem iguais às da inferência em lote.

In [ ]:
from hsi.paralelismo.fusao_concorrente import executar_concorrente, executar_sequencial, warm_pool
from hsi.segmentadores import segmenter_params

threads_antes = torch.get_num_threads()
# Uma thread do PyTorch por tarefa, como no experimento 08.
torch.set_num_threads(1)
dados = resultado["dados"]
entrada = {"pixels": pixels, "forma": gt.shape, "segmentador": config.segmenter,
           "params": segmenter_params(config), "seed": SEMENTE_DADOS, "coords": dados["coords"],
           "espectros": np.ascontiguousarray(dados["espectros"], dtype=np.float32)}
modelo = resultado["modelo"].to("cpu").eval()

lote = executar_sequencial(modelo, entrada)
print(f"lote, 1 thread: {1e3 * lote['parede']:.0f} ms (segmentação {1e3 * lote['t0_fim']:.0f} ms)")
for n in (2, 4, 8):
    pool = warm_pool(modelo, entrada, n)
    r = executar_concorrente(modelo, entrada, n, bloco=1024, pool=pool)
    pool.shutdown()
    iguais = np.array_equal(r["previsto"], lote["previsto"])
    print(f"concorrente, N={n}: {1e3 * r['parede']:.0f} ms, "
          f"speedup {lote['parede'] / r['parede']:.2f}x, classes iguais ao lote: {iguais}")
torch.set_num_threads(threads_antes)

## 7. Partição da cena

A cena dividida em N faixas, cada uma treinada e avaliada por conta própria, ao mesmo tempo. Com processos, cada parte roda num interpretador separado, livre do GIL.

In [ ]:
PARTICIONAR = False     # 4 partes, 25 épocas, cerca de 1 min por executor

if PARTICIONAR:
    from hsi.paralelismo.particao import run_experiment_particionado

    config_particao = ExperimentConfig(dataset="IP", epochs=25, batch_size=64)
    for executor in ("sequencial", "thread", "process"):
        r = run_experiment_particionado(config_particao, n=4, executor=executor)
        print(f"{executor:<10} bloco paralelo {r['tempo_paralelo']:6.1f} s   OA {r['OA']:.4f}")

## 8. Experimentos e figuras

As campanhas da monografia. Cada uma grava o CSV em `experimentos/resultados/` a cada medição e, se interrompida, continua de onde parou. Duram de minutos a horas; as longas rodam melhor no terminal, por exemplo `.venv/bin/python src/executar.py 03 IP`.

| Exp. | O que mede |
| --- | --- |
| 01 | linha de base, com a CNN do projeto e a do original |
| 02 | segmentadores com a configuração fixa |
| 03 | cada segmentador com os hiperparâmetros otimizados para ele |
| 04 | busca em grade contra otimização bayesiana |
| 05 | configuração otimizada contra a original |
| 06 | partição da cena |
| 07 | pool frio contra pool quente |
| 08 | fusão concorrente |

In [ ]:
EXECUTAR = None         # por exemplo "01"
CENAS = ("IP",)
GERAR_FIGURAS = False   # as figuras da monografia, a partir dos CSVs

if EXECUTAR is not None:
    from hsi.experimentos import EXPERIMENTOS

    torch.set_num_threads(1)     # como no terminal
    EXPERIMENTOS[EXECUTAR](*CENAS)
if GERAR_FIGURAS:
    from hsi.figuras import gerar

    gerar()